# 01 -- data generation (v2)

plays 5,000 games using adaptive expectimax search and records
(encoded_board, cumulative_score_to_end) pairs.

features in version 2:
- 5,000 full games simulated using parallel multiprocessing across cpu cores
- adaptive expectimax search: depth 2 during open boards (>= 6 empty), depth 3 during tight tactical phases (<= 5 empty)
- 8x dihedral symmetry augmentation (d4 group: 4 rotations x 2 reflections)
- board encoding: 16 floats, each cell = log2(value) / 17 (empty = 0)
- output: `training_data_v2.npz` packaged into `01_data_generation_v2_output.zip`

In [1]:
import numpy as np
import random
import math
import time
import os
import shutil
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import cpu_count

print(f'system cpu count: {cpu_count()}')

system cpu count: 4


## game engine

In [2]:
def clone(board):
    return [row[:] for row in board]

def get_empty_cells(board):
    return [(r, c) for r in range(4) for c in range(4) if board[r][c] == 0]

def spawn(board):
    b = clone(board)
    empty = get_empty_cells(b)
    if not empty:
        return b
    r, c = random.choice(empty)
    b[r][c] = 2 if random.random() < 0.9 else 4
    return b

def slide_row(row):
    score = 0
    vals = [v for v in row if v != 0]
    i = 0
    while i < len(vals) - 1:
        if vals[i] == vals[i + 1]:
            vals[i] *= 2
            score += vals[i]
            vals.pop(i + 1)
        i += 1
    while len(vals) < 4:
        vals.append(0)
    return vals, score

def move(board, direction):
    b = clone(board)
    total_score = 0
    moved = False
    for i in range(4):
        if direction == 0:   row = [b[0][i], b[1][i], b[2][i], b[3][i]]
        elif direction == 1: row = [b[i][3], b[i][2], b[i][1], b[i][0]]
        elif direction == 2: row = [b[3][i], b[2][i], b[1][i], b[0][i]]
        else:                row = [b[i][0], b[i][1], b[i][2], b[i][3]]
        new_row, score = slide_row(row)
        total_score += score
        for j in range(4):
            if direction == 0:   r2, c2 = j, i
            elif direction == 1: r2, c2 = i, 3 - j
            elif direction == 2: r2, c2 = 3 - j, i
            else:                r2, c2 = i, j
            if b[r2][c2] != new_row[j]:
                moved = True
            b[r2][c2] = new_row[j]
    return b, total_score, moved

def is_game_over(board):
    for r in range(4):
        for c in range(4):
            if board[r][c] == 0:
                return False
            if c < 3 and board[r][c] == board[r][c + 1]:
                return False
            if r < 3 and board[r][c] == board[r + 1][c]:
                return False
    return True

def create_board():
    board = [[0]*4 for _ in range(4)]
    board = spawn(board)
    board = spawn(board)
    return board

## heuristic evaluator and adaptive expectimax

In [3]:
PROB_THRESHOLD = 0.0001

def heuristic_evaluate(board):
    empty = 0
    max_val = 0
    mono = 0
    smooth = 0.0

    for r in range(4):
        for c in range(4):
            v = board[r][c]
            if v == 0:
                empty += 1
                continue
            log_v = math.log2(v)
            if v > max_val:
                max_val = v
            if c < 3 and board[r][c + 1] != 0:
                smooth -= abs(log_v - math.log2(board[r][c + 1]))
            if r < 3 and board[r + 1][c] != 0:
                smooth -= abs(log_v - math.log2(board[r + 1][c]))

    for r in range(4):
        inc = dec = 0
        for c in range(3):
            cur = 0 if board[r][c] == 0 else math.log2(board[r][c])
            nxt = 0 if board[r][c + 1] == 0 else math.log2(board[r][c + 1])
            if cur > nxt: dec += nxt - cur
            else:         inc += cur - nxt
        mono += max(inc, dec)

    for c in range(4):
        inc = dec = 0
        for r in range(3):
            cur = 0 if board[r][c] == 0 else math.log2(board[r][c])
            nxt = 0 if board[r + 1][c] == 0 else math.log2(board[r + 1][c])
            if cur > nxt: dec += nxt - cur
            else:         inc += cur - nxt
        mono += max(inc, dec)

    corners = [board[0][0], board[0][3], board[3][0], board[3][3]]
    corner = math.log2(max_val) if max_val in corners and max_val > 0 else 0

    return (empty * 2.7) + (mono * 1.0) + (smooth * 0.1) + (corner * 1.0)

def get_search_depth(board):
    # dynamic depth: depth 2 during open boards, depth 3 during tight tactical phases
    empty = len(get_empty_cells(board))
    return 3 if empty <= 5 else 2

def expectimax(board, depth, is_max, prob):
    if depth == 0 or is_game_over(board):
        return heuristic_evaluate(board)

    if is_max:
        best = float('-inf')
        for d in range(4):
            b, s, moved = move(board, d)
            if not moved:
                continue
            val = expectimax(b, depth - 1, False, prob)
            if val > best:
                best = val
        return best if best != float('-inf') else heuristic_evaluate(board)

    empty = get_empty_cells(board)
    if not empty:
        return heuristic_evaluate(board)

    avg = 0
    cell_prob = prob / len(empty)
    for r, c in empty:
        for val, tile_prob in [(2, 0.9), (4, 0.1)]:
            branch_prob = cell_prob * tile_prob
            if branch_prob < PROB_THRESHOLD:
                continue
            b = clone(board)
            b[r][c] = val
            avg += tile_prob * expectimax(b, depth - 1, True, branch_prob)
    return avg / len(empty)

def best_move(board):
    depth = get_search_depth(board)
    best_dir = -1
    best_val = float('-inf')
    for d in range(4):
        b, s, moved = move(board, d)
        if not moved:
            continue
        val = s + expectimax(b, depth - 1, False, 1.0)
        if val > best_val:
            best_val = val
            best_dir = d
    return best_dir

## board encoding and 8x symmetry augmentation

In [4]:
def encode_board(board):
    """encode a 4x4 board into 16 floats: log2(value)/17, empty=0."""
    encoded = []
    for r in range(4):
        for c in range(4):
            v = board[r][c]
            encoded.append(math.log2(v) / 17.0 if v > 0 else 0.0)
    return encoded

def augment_symmetries(boards, scores):
    """expand dataset 8x using d4 dihedral group symmetries (4 rotations x 2 reflections)."""
    b4 = boards.reshape(-1, 4, 4)
    aug_b = []
    aug_s = []

    for k in range(4):
        # rotate 90 * k degrees
        rot = np.rot90(b4, k, axes=(1, 2))
        aug_b.append(rot.reshape(-1, 16))
        aug_s.append(scores)

        # horizontal reflection of rotated board
        flip = np.flip(rot, axis=2)
        aug_b.append(flip.reshape(-1, 16))
        aug_s.append(scores)

    return np.concatenate(aug_b, axis=0), np.concatenate(aug_s, axis=0)

## simulation worker

In [5]:
def run_single_game(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32))

    board = create_board()
    game_boards = []
    game_move_scores = []
    total_score = 0

    while not is_game_over(board):
        game_boards.append(encode_board(board))
        game_move_scores.append(total_score)

        d = best_move(board)
        if d == -1:
            break
        board, score, moved = move(board, d)
        total_score += score
        board = spawn(board)

    # remaining score to end for each state
    labels = [total_score - s for s in game_move_scores]
    max_tile = max(max(row) for row in board)

    return game_boards, labels, total_score, max_tile

## execute 5,000 games in parallel

In [6]:
NUM_GAMES = 5000
NUM_WORKERS = max(1, cpu_count())
print(f'starting parallel simulation: {NUM_GAMES} games across {NUM_WORKERS} worker processes...')

all_boards = []
all_scores = []
game_stats = []

t_start = time.time()
completed = 0

# use process pool executor for true multi-core parallel simulation
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = [executor.submit(run_single_game, 100000 + i) for i in range(NUM_GAMES)]

    for future in as_completed(futures):
        boards, scores, total_score, max_tile = future.result()
        all_boards.extend(boards)
        all_scores.extend(scores)
        game_stats.append((total_score, max_tile))
        completed += 1

        if completed % 250 == 0 or completed == NUM_GAMES:
            elapsed = time.time() - t_start
            rate = completed / elapsed
            eta_sec = (NUM_GAMES - completed) / rate if rate > 0 else 0
            avg_score = sum(s for s, _ in game_stats[-250:]) / min(completed, 250)
            print(f'games: {completed:4d}/{NUM_GAMES} | avg score: {avg_score:.0f} | '
                  f'samples: {len(all_boards):7d} | rate: {rate:.1f} games/s | '
                  f'elapsed: {elapsed:.0f}s | eta: {eta_sec:.0f}s')

elapsed = time.time() - t_start
print(f'\nsimulation complete in {elapsed:.1f}s ({elapsed/60:.1f} minutes).')
print(f'total raw samples collected: {len(all_boards):,}')
print(f'average game score: {sum(s for s, _ in game_stats) / NUM_GAMES:.0f}')

tile_counts = {}
for _, mt in game_stats:
    tile_counts[mt] = tile_counts.get(mt, 0) + 1
print(f'max tile distribution: {dict(sorted(tile_counts.items()))}')

starting parallel simulation: 5000 games across 4 worker processes...
games:  250/5000 | avg score: 21096 | samples:  289763 | rate: 1.0 games/s | elapsed: 259s | eta: 4918s
games:  500/5000 | avg score: 20653 | samples:  574074 | rate: 1.0 games/s | elapsed: 507s | eta: 4561s
games:  750/5000 | avg score: 20815 | samples:  861257 | rate: 1.0 games/s | elapsed: 758s | eta: 4297s
games: 1000/5000 | avg score: 21042 | samples: 1149946 | rate: 1.0 games/s | elapsed: 1011s | eta: 4046s
games: 1250/5000 | avg score: 21675 | samples: 1445886 | rate: 1.0 games/s | elapsed: 1271s | eta: 3812s
games: 1500/5000 | avg score: 21812 | samples: 1744090 | rate: 1.0 games/s | elapsed: 1534s | eta: 3580s
games: 1750/5000 | avg score: 20897 | samples: 2031587 | rate: 1.0 games/s | elapsed: 1782s | eta: 3310s
games: 2000/5000 | avg score: 21584 | samples: 2325200 | rate: 1.0 games/s | elapsed: 2037s | eta: 3055s
games: 2250/5000 | avg score: 21701 | samples: 2620452 | rate: 1.0 games/s | elapsed: 2294s |

## apply 8x dihedral symmetry augmentation

In [7]:
print('converting raw boards and scores to numpy arrays...')
raw_boards_arr = np.array(all_boards, dtype=np.float32)
raw_scores_arr = np.array(all_scores, dtype=np.float32)

print(f'raw boards shape: {raw_boards_arr.shape}')
print(f'raw scores shape: {raw_scores_arr.shape}')
print(f'score range: {raw_scores_arr.min():.0f} - {raw_scores_arr.max():.0f}')

t_aug_start = time.time()
print('applying 8x dihedral symmetry augmentation (rotations + reflections)...')
boards_arr, scores_arr = augment_symmetries(raw_boards_arr, raw_scores_arr)
t_aug_end = time.time()

print(f'augmented boards shape: {boards_arr.shape}')
print(f'augmented scores shape: {scores_arr.shape}')
print(f'augmentation completed in {t_aug_end - t_aug_start:.2f}s')

converting raw boards and scores to numpy arrays...
raw boards shape: (5875477, 16)
raw scores shape: (5875477,)
score range: 0 - 76564
applying 8x dihedral symmetry augmentation (rotations + reflections)...
augmented boards shape: (47003816, 16)
augmented scores shape: (47003816,)
augmentation completed in 2.56s


## save compressed dataset

In [8]:
OUTPUT_FILE = 'training_data_v2.npz'
print(f'saving compressed dataset to {OUTPUT_FILE}...')
np.savez_compressed(OUTPUT_FILE, boards=boards_arr, scores=scores_arr)

size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f'successfully saved {OUTPUT_FILE} ({size_mb:.1f} MB)')

saving compressed dataset to training_data_v2.npz...
successfully saved training_data_v2.npz (359.5 MB)


## package output for download

In [9]:
os.makedirs('output', exist_ok=True)
shutil.copy('training_data_v2.npz', 'output/training_data_v2.npz')
shutil.make_archive('01_data_generation_v2_output', 'zip', 'output')

zip_size_mb = os.path.getsize('01_data_generation_v2_output.zip') / (1024 * 1024)
print(f'done. download 01_data_generation_v2_output.zip ({zip_size_mb:.1f} MB)')

done. download 01_data_generation_v2_output.zip (344.7 MB)
